# Hierarchical readmissions: the patient-level model and `funnel()`

This notebook fits physician 30-day readmissions against an external expected
(Vizient-style, one expected probability per stay) two ways:

1. **The full patient-level hierarchical model** in PyMC, the approach of the CMS
   hospital readmission measures (Keenan et al. 2008), applied to physicians:
   `logit(P) = logit(expected) + a + b_doc`, `b_doc ~ Normal(0, sigma)`. Use it when you
   have your own risk model or covariates to add.
2. **`qikit.funnel(compare_to="model")`**, which gives the same kind of output (a shrunk
   P/E per physician with a 95% interval and P(worse than expected)) from provider
   totals in a second, with no sampler.

It then compares the two, and adds an optional third level for hospital or campus,
because much of the variation among hospitalists belongs to the hospital (Goodwin et
al. 2013).

**Requirements.** PyMC is not a qikit dependency. Install it separately:
`pip install pymc clinical-qikit[plot]`. Written against PyMC 5.16.

Applying this approach at physician level, with an external expected as the
baseline, adapts the CMS hospital-level method. It applies established methods; it is
not a replication of a published physician-level design (docs/methods.md §4.14).

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
from qikit import funnel

rng = np.random.default_rng(2026)

## Simulated stays

6 hospitals and 90 physicians, each physician practising at one hospital. Every
stay has an expected probability from the external model (about 13% on average). The
system runs slightly above the model (`a`), hospitals differ a little (`sigma_hosp`),
physicians differ within hospitals (`sigma_doc`), and four physicians are truly worse.
Replace this cell with your own stay-level query (one row per stay: physician,
hospital, readmitted 0/1, expected probability).

In [ ]:
n_hosp, n_doc = 6, 90
a_true, sigma_hosp, sigma_doc = 0.08, 0.10, 0.20
hosp_of_doc = rng.integers(0, n_hosp, n_doc)
c_hosp = rng.normal(0, sigma_hosp, n_hosp)
b_doc = rng.normal(0, sigma_doc, n_doc)
b_doc[:4] += 0.6                                   # four truly worse physicians

rows = []
for d in range(n_doc):
    for _ in range(rng.integers(30, 300)):
        p_exp = float(np.clip(rng.beta(2.0, 13.4), 0.01, 0.9))
        eta = np.log(p_exp / (1 - p_exp)) + a_true + c_hosp[hosp_of_doc[d]] + b_doc[d]
        rows.append((f"Dr {d:02d}", f"Hospital {hosp_of_doc[d]}", p_exp,
                     int(rng.random() < 1 / (1 + np.exp(-eta)))))
stays = pd.DataFrame(rows, columns=["doc", "hospital", "expected", "readmit"])
stays.groupby("doc").agg(stays=("readmit", "size"), observed=("readmit", "sum"),
                         expected=("expected", "sum")).describe().round(1)

## 1. The patient-level model in PyMC

Non-centered parameterization for the physician effects. The P/E for a physician is
the predicted readmissions under their own effect, summed over their stays, divided
by the external expected. A ratio above 1 means more readmissions than the risk
model expects.

In [ ]:
docs = sorted(stays["doc"].unique())
doc_idx = stays["doc"].map({d: i for i, d in enumerate(docs)}).to_numpy()
logit_exp = np.log(stays["expected"] / (1 - stays["expected"])).to_numpy()
y = stays["readmit"].to_numpy()

with pm.Model() as two_level:
    a = pm.Normal("a", 0.0, 1.0)
    sigma = pm.HalfNormal("sigma", 0.5)
    z = pm.Normal("z", 0.0, 1.0, shape=len(docs))
    b = pm.Deterministic("b", sigma * z)
    pm.Bernoulli("y", logit_p=logit_exp + a + b[doc_idx], observed=y)
    idata = pm.sample(1000, tune=1000, chains=4, target_accept=0.9, random_seed=1,
                      progressbar=False)

In [ ]:
def p_over_e(idata, doc_idx, logit_exp, expected, extra=0.0):
    """Posterior draws of each physician's P/E against the external expected."""
    post = idata.posterior
    a = post["a"].values.reshape(-1)
    b = post["b"].values.reshape(len(a), -1)
    eta = logit_exp[None, :] + a[:, None] + b[:, doc_idx] + extra
    pred = 1 / (1 + np.exp(-eta))
    n_doc = b.shape[1]
    p_sum = np.zeros((len(a), n_doc))
    np.add.at(p_sum.T, doc_idx, pred.T)
    e_sum = np.bincount(doc_idx, weights=expected, minlength=n_doc)
    return p_sum / e_sum

draws = p_over_e(idata, doc_idx, logit_exp, stays["expected"].to_numpy())
pymc_table = pd.DataFrame({
    "doc": docs,
    "pymc_pe": draws.mean(0),
    "pymc_low": np.quantile(draws, 0.025, axis=0),
    "pymc_high": np.quantile(draws, 0.975, axis=0),
    "pymc_prob_worse": (draws > 1.0).mean(0),
})
print("sigma (doc SD, logit scale):", float(idata.posterior["sigma"].mean()))
pymc_table.sort_values("pymc_pe", ascending=False).head(8).round(3)

## 2. The same question with `funnel()`

Stay-level rows are summed per physician. `compare_to="model"` judges each physician
against 1.0 (observed equals the external expected). The estimate columns shrink
toward the system's pooled value with the spread estimated by maximum likelihood
(docs/methods.md §4.14).

In [ ]:
r = funnel(stays, y="readmit", n="expected", measure="oe", unit="doc",
           compare_to="model", worse="higher")
qikit_table = r.data[["unit", "y", "eb_rate", "eb_low", "eb_high", "prob_worse",
                      "estimate_flag", "signal", "reliability"]].rename(columns={"unit": "doc"})
print(r)
qikit_table.sort_values("eb_rate", ascending=False).head(8).round(3)

In [ ]:
r.plot(kind="caterpillar", title="Physician readmissions against the external expected")

## 3. How close are they?

Close but not identical, for three reasons. PyMC's effects are on the logit scale and
qikit's on the O/E (Poisson) scale; PyMC integrates over the uncertainty in `sigma`
and `a`, where qikit plugs in estimates; and PyMC uses each stay's own expected
probability, qikit only the totals.

In [ ]:
both = pymc_table.merge(qikit_table, on="doc")
diff = pd.DataFrame({
    "estimate": (both["eb_rate"] - both["pymc_pe"]).abs(),
    "low": (both["eb_low"] - both["pymc_low"]).abs(),
    "high": (both["eb_high"] - both["pymc_high"]).abs(),
    "prob_worse": (both["prob_worse"] - both["pymc_prob_worse"]).abs(),
})
print("correlation of estimates:", round(both["eb_rate"].corr(both["pymc_pe"]), 4))
print("interval width ratio (qikit / PyMC), median:",
      round(float(np.median((both.eb_high - both.eb_low) / (both.pymc_high - both.pymc_low))), 3))
print("same 'worse' call (interval wholly above 1):",
      round(float(((both.eb_low > 1) == (both.pymc_low > 1)).mean()), 3))
diff.describe().loc[["mean", "50%", "max"]].round(3)

**What one run found** (PyMC 5.16.2, the seeds above, 90 physicians, about 15,000
stays): the estimates correlate at 0.996; qikit's intervals are about 4% narrower
(median), as expected from plugging in the spread rather than integrating over it; the
two agree on which physicians' intervals lie wholly above 1.0 for 96% of physicians;
and the median absolute difference is 0.03 in the estimate and 0.01 in P(worse). The
largest differences are low-volume physicians with extreme raw values, whom the
gamma prior on the O/E scale shrinks harder than the normal prior on the logit scale.
Re-run with your own data before relying on the comparison.

## 4. Optional: a third level for hospital or campus

Add a hospital effect so that a physician is compared with what their hospital's
patients would be expected to do there. Physician effects then measure variation
within hospitals. The `funnel()` analogue is `facets="hospital"`: one chart per
hospital, each with its own pooled value and spread (with few physicians per hospital,
pass a `tau=` from earlier periods).

In [ ]:
hosps = sorted(stays["hospital"].unique())
hosp_idx_doc = np.array([hosps.index(stays.loc[stays.doc == d, "hospital"].iloc[0])
                         for d in docs])

with pm.Model() as three_level:
    a = pm.Normal("a", 0.0, 1.0)
    sigma_h = pm.HalfNormal("sigma_hosp", 0.5)
    sigma_d = pm.HalfNormal("sigma", 0.5)
    c = pm.Deterministic("c", sigma_h * pm.Normal("zc", 0.0, 1.0, shape=len(hosps)))
    b = pm.Deterministic("b", sigma_d * pm.Normal("z", 0.0, 1.0, shape=len(docs)))
    eta = logit_exp + a + c[hosp_idx_doc[doc_idx]] + b[doc_idx]
    pm.Bernoulli("y", logit_p=eta, observed=y)
    idata3 = pm.sample(1000, tune=1000, chains=4, target_accept=0.95, random_seed=2,
                       progressbar=False)

print("sigma_hosp:", float(idata3.posterior["sigma_hosp"].mean()),
      " sigma_doc:", float(idata3.posterior["sigma"].mean()))

In [ ]:
by_hospital = funnel(stays, y="readmit", n="expected", measure="oe", unit="doc",
                     facets="hospital", compare_to="model", worse="higher")
by_hospital.plot(kind="caterpillar", top=5, bottom=3)